# Fraud Detection — Data Preprocessing

Preprocessing pipeline for `transactions.csv`, tailored for tree-based models:
- **XGBoost**, **Random Forest**, **Decision Trees**

### Preprocessing Strategy
| Feature Type | Method |
|---|---|
| Skewed numeric (`amt`, `city_pop`) | Log-transform |
| Ordinal categorical (`income_band`, `education`, `merchant_size`) | Ordinal integer encoding |
| Nominal categorical (`mcc`, `state`, `job`, `gender`, `marital_status`) | One-hot encoding |
| Date (`dob`) | Convert to age (years) |
| Identifier (`cc_num`, `city`) | Drop |
| Other numeric | Passthrough |

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer, OrdinalEncoder, OneHotEncoder
)

SEED = 42
TEST_SIZE = 0.2

## 1. Load Data & Select Features

In [9]:
DATA_PATH = "Sparkov_Data_Generation/output_data/transactions.csv"

# All columns we need from the CSV
USE_COLS = [
    "unix_time", "transaction_velocity", "amt", "mcc",
    "lat", "long", "merch_lat", "merch_long",
    "cc_num", "dob", "gender", "job",
    "income_band", "education", "marital_status",
    "city", "state", "city_pop", "merchant_size",
    "is_fraud",
]

df = pd.read_csv(DATA_PATH, sep="|", usecols=USE_COLS)
print(f"{len(df):,} rows | {df['is_fraud'].sum():,} fraud ({df['is_fraud'].mean():.3%} base rate)")
df.head()

891,174 rows | 5,023 fraud (0.564% base rate)


,cc_num,gender,city,state,lat,long,city_pop,job,dob,income_band,education,marital_status,unix_time,amt,is_fraud,merch_lat,merch_long,mcc,merchant_size,transaction_velocity
0,5332871012269166,M,Newport News,VA,37.1989,-76.5821,179275,Professor,1971-08-05,100k+,bachelor_plus,married,1767345928,93.00,0,37.554566,-77.158093,5812,large,0
1,5332871012269166,M,Newport News,VA,37.1989,-76.5821,179275,Professor,1971-08-05,100k+,bachelor_plus,married,1767751202,62.38,0,37.686739,-77.097054,5812,large,0
2,5332871012269166,M,Newport News,VA,37.1989,-76.5821,179275,Professor,1971-08-05,100k+,bachelor_plus,married,1767783881,41.87,0,36.747615,-75.979476,5812,small,1
3,5332871012269166,M,Newport News,VA,37.1989,-76.5821,179275,Professor,1971-08-05,100k+,bachelor_plus,married,1768147920,90.64,0,36.687107,-76.373284,5812,medium,0
4,5332871012269166,M,Newport News,VA,37.1989,-76.5821,179275,Professor,1971-08-05,100k+,bachelor_plus,married,1768237814,7.35,0,36.810673,-77.031967,5812,small,0


## 2. Feature Engineering

Before the pipeline, we handle transformations that create or remove columns.

In [10]:
# --- Convert dob -> age (in years, relative to the transaction time) ---
df["dob"] = pd.to_datetime(df["dob"])
# Use the median transaction date as reference point
reference_date = pd.to_datetime(df["unix_time"].median(), unit="s")
df["age"] = (reference_date - df["dob"]).dt.days / 365.25
df["age"] = df["age"].round(1)

print(f"Age range: {df['age'].min():.1f} – {df['age'].max():.1f} years")
print(f"Age mean:  {df['age'].mean():.1f}, std: {df['age'].std():.1f}")

# --- Drop columns we don't need ---
# cc_num: identifier, not a feature
# city: too many unique values (438), risk of overfitting
# dob: replaced by age
df = df.drop(columns=["cc_num", "city", "dob"])

# --- Cast mcc to string (it's a category code, not a number) ---
df["mcc"] = df["mcc"].astype(str)

print(f"\nFinal columns ({len(df.columns)}): {list(df.columns)}")
df.head()

Age range: 17.6 – 85.5 years
Age mean:  42.8, std: 17.6

Final columns (18): ['gender', 'state', 'lat', 'long', 'city_pop', 'job', 'income_band', 'education', 'marital_status', 'unix_time', 'amt', 'is_fraud', 'merch_lat', 'merch_long', 'mcc', 'merchant_size', 'transaction_velocity', 'age']


,gender,state,lat,long,city_pop,job,income_band,education,marital_status,unix_time,amt,is_fraud,merch_lat,merch_long,mcc,merchant_size,transaction_velocity,age
0,M,VA,37.1989,-76.5821,179275,Professor,100k+,bachelor_plus,married,1767345928,93.00,0,37.554566,-77.158093,5812,large,0,55.0
1,M,VA,37.1989,-76.5821,179275,Professor,100k+,bachelor_plus,married,1767751202,62.38,0,37.686739,-77.097054,5812,large,0,55.0
2,M,VA,37.1989,-76.5821,179275,Professor,100k+,bachelor_plus,married,1767783881,41.87,0,36.747615,-75.979476,5812,small,1,55.0
3,M,VA,37.1989,-76.5821,179275,Professor,100k+,bachelor_plus,married,1768147920,90.64,0,36.687107,-76.373284,5812,medium,0,55.0
4,M,VA,37.1989,-76.5821,179275,Professor,100k+,bachelor_plus,married,1768237814,7.35,0,36.810673,-77.031967,5812,small,0,55.0


## 3. Train/Test Split

Stratified split to preserve the fraud rate in both sets.

In [11]:
TARGET = "is_fraud"
y = df[TARGET]
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Train: {len(X_train):,} rows ({y_train.mean():.3%} fraud)")
print(f"Test:  {len(X_test):,} rows  ({y_test.mean():.3%} fraud)")

Train: 712,939 rows (0.564% fraud)
Test:  178,235 rows  (0.564% fraud)


## 4. Preprocessing Pipeline

### Strategy by feature group:

| Group | Features | Transform | Rationale |
|-------|----------|-----------|----------|
| **Log-transform** | `amt`, `city_pop` | `log1p` | Heavily right-skewed (skewness > 1.4), log spreads out clustered values for better tree splits |
| **Passthrough** | `unix_time`, `transaction_velocity`, `lat`, `long`, `merch_lat`, `merch_long`, `age` | None | Moderate variance/skewness, trees handle fine as-is |
| **Ordinal encode** | `income_band`, `education`, `merchant_size` | Integer mapping (0, 1, 2) | Ranked features where order matters |
| **One-hot encode** | `mcc`, `state`, `job`, `gender`, `marital_status` | Binary columns | Nominal categories with no meaningful order |

In [12]:
# ============================================================
# Define column groups
# ============================================================

# Skewed numeric -> log-transform
LOG_COLS = ["amt", "city_pop"]

# Numeric -> passthrough (no scaling needed for tree models)
PASSTHROUGH_COLS = [
    "unix_time", "transaction_velocity",
    "lat", "long", "merch_lat", "merch_long",
    "age",
]

# Ordinal categoricals -> integer encoding preserving rank
ORDINAL_COLS = ["income_band", "education", "merchant_size"]
ORDINAL_CATEGORIES = [
    ["<50k", "50-100k", "100k+"],                       # income_band: low -> high
    ["hs_or_less", "some_college", "bachelor_plus"],     # education: low -> high
    ["small", "medium", "large"],                        # merchant_size: small -> large
]

# Nominal categoricals -> one-hot encoding
ONEHOT_COLS = ["mcc", "state", "job", "gender", "marital_status"]

print(f"Log-transform:    {len(LOG_COLS)} cols  -> {LOG_COLS}")
print(f"Passthrough:      {len(PASSTHROUGH_COLS)} cols  -> {PASSTHROUGH_COLS}")
print(f"Ordinal encode:   {len(ORDINAL_COLS)} cols  -> {ORDINAL_COLS}")
print(f"One-hot encode:   {len(ONEHOT_COLS)} cols  -> {ONEHOT_COLS}")

Log-transform:    2 cols  -> ['amt', 'city_pop']
Passthrough:      7 cols  -> ['unix_time', 'transaction_velocity', 'lat', 'long', 'merch_lat', 'merch_long', 'age']
Ordinal encode:   3 cols  -> ['income_band', 'education', 'merchant_size']
One-hot encode:   5 cols  -> ['mcc', 'state', 'job', 'gender', 'marital_status']


In [13]:
# ============================================================
# Build the ColumnTransformer
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        # 1. Log-transform skewed numeric features
        #    Using log1p (log(1+x)) to handle zeros safely
        #    feature_names_out='one-to-one' allows get_feature_names_out() to work
        ("log", FunctionTransformer(np.log1p, validate=True, feature_names_out="one-to-one"), LOG_COLS),

        # 2. Passthrough numeric features (no transformation)
        ("num", "passthrough", PASSTHROUGH_COLS),

        # 3. Ordinal encoding for ranked categorical features
        ("ord", OrdinalEncoder(categories=ORDINAL_CATEGORIES,
                               handle_unknown="use_encoded_value",
                               unknown_value=-1),
         ORDINAL_COLS),

        # 4. One-hot encoding for nominal categorical features
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         ONEHOT_COLS),
    ],
    remainder="drop",  # Drop any column not explicitly listed
    verbose_feature_names_out=True,
)

print("Pipeline configured. Fitting on training data...")

Pipeline configured. Fitting on training data...


In [14]:
# ============================================================
# Fit on training data, transform both sets
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names for interpretability
feature_names = preprocessor.get_feature_names_out()

# Convert to DataFrames for inspection
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print(f"Processed feature count: {X_train_df.shape[1]}")
print(f"Train shape: {X_train_df.shape}")
print(f"Test shape:  {X_test_df.shape}")
print(f"\nFeature names ({len(feature_names)}):\n{list(feature_names)}")

Processed feature count: 98
Train shape: (712939, 98)
Test shape:  (178235, 98)

Feature names (98):
['log__amt', 'log__city_pop', 'num__unix_time', 'num__transaction_velocity', 'num__lat', 'num__long', 'num__merch_lat', 'num__merch_long', 'num__age', 'ord__income_band', 'ord__education', 'ord__merchant_size', 'ohe__mcc_4722', 'ohe__mcc_4900', 'ohe__mcc_5311', 'ohe__mcc_5411', 'ohe__mcc_5712', 'ohe__mcc_5812', 'ohe__mcc_5912', 'ohe__mcc_5945', 'ohe__mcc_5999', 'ohe__mcc_7230', 'ohe__state_AK', 'ohe__state_AL', 'ohe__state_AR', 'ohe__state_AZ', 'ohe__state_CA', 'ohe__state_CO', 'ohe__state_CT', 'ohe__state_DC', 'ohe__state_FL', 'ohe__state_GA', 'ohe__state_HI', 'ohe__state_IA', 'ohe__state_ID', 'ohe__state_IL', 'ohe__state_IN', 'ohe__state_KS', 'ohe__state_KY', 'ohe__state_LA', 'ohe__state_MA', 'ohe__state_MD', 'ohe__state_ME', 'ohe__state_MI', 'ohe__state_MN', 'ohe__state_MO', 'ohe__state_MS', 'ohe__state_MT', 'ohe__state_NC', 'ohe__state_ND', 'ohe__state_NE', 'ohe__state_NH', 'ohe__st

## 5. Verify Preprocessing

In [15]:
# ============================================================
# Verify log-transform reduced skewness
# ============================================================

print("=== Skewness Comparison (before vs after log-transform) ===")
for col in LOG_COLS:
    before = X_train[col].skew()
    log_col_name = f"log__{col}"
    after = X_train_df[log_col_name].skew()
    print(f"  {col:15s}  before: {before:+.3f}  ->  after: {after:+.3f}")

print("\n=== Ordinal Encoding Verification ===")
for col, cats in zip(ORDINAL_COLS, ORDINAL_CATEGORIES):
    ord_col = f"ord__{col}"
    print(f"  {col}: {cats} -> mapped to {sorted(X_train_df[ord_col].unique())}")

print(f"\n=== One-Hot Columns Created ===")
ohe_cols = [c for c in feature_names if c.startswith("ohe__")]
print(f"  {len(ohe_cols)} one-hot columns")
print(f"  Examples: {ohe_cols[:6]}...")

=== Skewness Comparison (before vs after log-transform) ===
  amt              before: +1.616  ->  after: -0.358
  city_pop         before: +2.416  ->  after: -1.315

=== Ordinal Encoding Verification ===
  income_band: ['<50k', '50-100k', '100k+'] -> mapped to [np.float64(0.0), np.float64(1.0), np.float64(2.0)]
  education: ['hs_or_less', 'some_college', 'bachelor_plus'] -> mapped to [np.float64(0.0), np.float64(1.0), np.float64(2.0)]
  merchant_size: ['small', 'medium', 'large'] -> mapped to [np.float64(0.0), np.float64(1.0), np.float64(2.0)]

=== One-Hot Columns Created ===
  86 one-hot columns
  Examples: ['ohe__mcc_4722', 'ohe__mcc_4900', 'ohe__mcc_5311', 'ohe__mcc_5411', 'ohe__mcc_5712', 'ohe__mcc_5812']...


In [16]:
# ============================================================
# Summary statistics of processed features
# ============================================================

# Show stats for the non-OHE columns (more interpretable)
non_ohe_cols = [c for c in feature_names if not c.startswith("ohe__")]
print("=== Processed Feature Statistics (non-OHE columns) ===")
X_train_df[non_ohe_cols].describe().round(3)

=== Processed Feature Statistics (non-OHE columns) ===


,log__amt,log__city_pop,num__unix_time,num__transaction_velocity,num__lat,num__long,num__merch_lat,num__merch_long,num__age,ord__income_band,ord__education,ord__merchant_size
count,712939.000,712939.000,7.129390e+05,712939.000,712939.000,712939.000,712939.000,712939.000,712939.000,712939.000,712939.000,712939.000
mean,4.046,11.595,1.784629e+09,4.097,37.735,-93.273,37.735,-93.273,42.842,0.877,1.150,0.702
std,1.259,2.339,9.031677e+06,3.311,5.419,15.557,5.449,15.567,17.628,0.809,0.832,0.806
min,0.693,2.639,1.767254e+09,0.000,21.312,-158.017,20.318,-159.015,17.600,0.000,0.000,0.000
25%,3.348,10.964,1.777238e+09,2.000,33.741,-98.459,33.675,-98.368,29.200,0.000,0.000,0.000
50%,4.170,12.117,1.784769e+09,3.000,38.235,-90.208,38.249,-90.386,39.000,1.000,1.000,0.000
75%,4.627,13.244,1.792659e+09,6.000,40.931,-81.569,41.051,-81.686,53.500,2.000,2.000,1.000
max,6.217,14.883,1.798790e+09,34.000,64.859,-67.715,65.859,-66.715,85.500,2.000,2.000,2.000


In [17]:
# ============================================================
# Check for nulls/NaNs
# ============================================================

null_counts = X_train_df.isnull().sum()
if null_counts.sum() == 0:
    print("No missing values in processed features")
else:
    print("Missing values found:")
    print(null_counts[null_counts > 0])

No missing values in processed features


## 6. Export Preprocessed Data

Save processed data and the fitted preprocessor for use in modeling.

In [18]:
import joblib
import os

OUTPUT_DIR = "preprocessed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save processed DataFrames
X_train_df.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test_df.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

# Save the fitted preprocessor (for applying to new data later)
joblib.dump(preprocessor, f"{OUTPUT_DIR}/preprocessor.joblib")

print(f"Saved to {OUTPUT_DIR}/")
print(f"   X_train.csv  ({X_train_df.shape})")
print(f"   X_test.csv   ({X_test_df.shape})")
print(f"   y_train.csv  ({len(y_train):,} rows)")
print(f"   y_test.csv   ({len(y_test):,} rows)")
print(f"   preprocessor.joblib")

Saved to preprocessed_data/
   X_train.csv  ((712939, 98))
   X_test.csv   ((178235, 98))
   y_train.csv  (712,939 rows)
   y_test.csv   (178,235 rows)
   preprocessor.joblib


In [19]:
# ============================================================
# Quick peek at final processed data
# ============================================================

print(f"Final processed training data: {X_train_df.shape[0]:,} rows x {X_train_df.shape[1]} features")
print(f"\nFirst 5 rows (all columns):")
X_train_df.head()

Final processed training data: 712,939 rows x 98 features

First 5 rows (all columns):


,log__amt,log__city_pop,num__unix_time,num__transaction_velocity,num__lat,num__long,num__merch_lat,num__merch_long,num__age,ord__income_band,...,ohe__job_Retired,ohe__job_Sales Associate,ohe__job_Software Engineer,ohe__job_Teacher,ohe__job_Unemployed,ohe__gender_F,ohe__gender_M,ohe__marital_status_formerly_married,ohe__marital_status_married,ohe__marital_status_never_married
645633,5.836972,11.344779,1.780719e+09,4.0,36.8668,-76.5598,37.438453,-76.243589,27.3,2.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
743935,4.168679,6.426488,1.791388e+09,0.0,46.3679,-112.0232,45.917587,-111.825245,32.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
201185,3.045474,11.665870,1.792855e+09,2.0,38.2351,-81.5536,37.395263,-81.013595,54.1,2.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
472553,2.356126,11.667268,1.798615e+09,6.0,47.6404,-92.4428,47.810866,-91.455660,48.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
848453,4.011687,11.878686,1.786334e+09,8.0,32.3766,-95.1252,33.301399,-94.773648,24.1,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
